# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
# The raw file contains non-UTF-8 characters, so it is read with latin-1 encoding
df = pd.read_csv('data/AviationData.csv', encoding='latin-1', low_memory=False)
print(df.shape)
df.head()

(88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [3]:
# Data types and non-null counts for every column
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [4]:
# Number and percentage of missing values per column, most missing first
nan_summary = pd.DataFrame({'n_missing': df.isna().sum(),
                            'pct_missing': (df.isna().mean() * 100).round(1)})
nan_summary.sort_values('pct_missing', ascending=False)

,n_missing,pct_missing
Schedule,76307,85.8
Air.carrier,72241,81.3
FAR.Description,56866,64.0
Aircraft.Category,56602,63.7
Longitude,54516,61.3
Latitude,54507,61.3
Airport.Code,38757,43.6
Airport.Name,36185,40.7
Broad.phase.of.flight,27165,30.6
Publication.Date,13771,15.5


In [5]:
# Summary statistics for the numeric columns
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


In [6]:
# Summary statistics for the text (categorical) columns
df.describe(include='object').T

/tmp/ipykernel_267/3312589043.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include='object').T


,count,unique,top,freq
Event.Id,88889,87951,20001214X45071,3
Investigation.Type,88889,2,Accident,85015
Accident.Number,88889,88863,ERA22LA103,2
Event.Date,88889,14782,1982-05-16,25
Location,88837,27758,"ANCHORAGE, AK",434
Country,88663,219,United States,82248
Latitude,34382,25589,332739N,19
Longitude,34373,27154,0112457W,24
Airport.Code,50132,10374,NONE,1488
Airport.Name,52704,24870,Private,240


**Inspection notes**
- The data has 88,889 events and 31 columns.
- `Event.Date` is stored as text and needs converting to a date so the data can be filtered by year.
- `Aircraft.Category`, `FAR.Description`, `Schedule` and `Air.carrier` are missing for most rows.
- The injury columns (`Total.Fatal.Injuries` etc.) are missing for roughly 10–14% of rows.
- `Make` mixes upper and lower case (e.g. `CESSNA` and `Cessna`), so names need standardizing.

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [7]:
# Convert the event date and extract the year
df['Event.Date'] = pd.to_datetime(df['Event.Date'])
df['Year'] = df['Event.Date'].dt.year
df['Year'].describe()

count    88889.000000
mean      1999.206662
std         11.888226
min       1948.000000
25%       1989.000000
50%       1998.000000
75%       2009.000000
max       2022.000000
Name: Year, dtype: float64

In [8]:
# Relevant filtering columns
for col in ['Aircraft.Category', 'Amateur.Built']:
    print(df[col].value_counts(dropna=False), '\n')

Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64 

Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int64 



**Filtering decisions**
1. **Active aircraft only:** with a maximum 40-year service life, keep events from **1983 onwards**.
2. **Professional builds only:** keep rows where `Amateur.Built` is **No** (homebuilt aircraft are excluded).
3. **Airplanes only:** the client insures airplanes, so helicopters, gliders, balloons etc. are removed.

`Aircraft.Category` is missing for about 64% of rows, so dropping all of those would throw away most of the data. **Reasonable imputation:** for each manufacturer we look at the rows where the category *is* known. If at least 90% of a manufacturer's known rows are airplanes (and it has at least 5 known rows), its missing categories are filled in as `Airplane`. Manufacturers that mainly build helicopters (e.g. Robinson, Bell) are therefore not mislabelled.

Because this imputation is done by manufacturer, the `Make` names are standardized first (see the Make section for the full list of cleaning steps).

In [9]:
# Filter to events from 1983 onwards and to professionally built aircraft
df = df[(df['Year'] >= 1983) & (df['Amateur.Built'] == 'No')].copy()

# Rows without a Make cannot be attributed to a manufacturer, so they are dropped
df = df.dropna(subset=['Make'])
print(df.shape)

(76914, 32)


In [10]:
import re

# Standardize Make names: consistent capitalization and removal of company suffixes
SUFFIX = r'\b(Inc|Corp|Corporation|Co|Company|Ltd|Llc|Aircraft|Airplane|Airplanes|Aviation|Industries|Mfg|Manufacturing|Design|Aerospace|Div|Avn)\b'

def clean_make(make):
    make = make.strip().title()            # 'CESSNA' and 'cessna' -> 'Cessna'
    make = re.sub(r'[\.,]', ' ', make)     # remove punctuation such as 'Co.' or 'Inc,'
    make = re.sub(SUFFIX, ' ', make)        # remove company suffixes
    return re.sub(r'\s+', ' ', make).strip()

# Remaining spelling variants of the same manufacturer
MAKE_FIXES = {'Dehavilland': 'De Havilland', 'Beechcraft': 'Beech', 'Airbus Industrie': 'Airbus',
              'Embraer S A': 'Embraer', 'Embraer-Empresa Brasileira De': 'Embraer', 'New Piper': 'Piper',
              'Gates Learjet': 'Learjet', 'Grumman American Avn': 'Grumman American',
              'British': 'British Aerospace', 'Diamond Ind': 'Diamond',
              'Grumman Acft Eng Cor-Schweizer': 'Grumman-Schweizer', 'Christen Ind': 'Christen'}

df['Make'] = df['Make'].map(clean_make).replace(MAKE_FIXES)
print('Unique makes after standardizing:', df['Make'].nunique())

Unique makes after standardizing: 1558


In [11]:
# Impute missing Aircraft.Category using each manufacturer's known categories
known = df.dropna(subset=['Aircraft.Category'])
airplane_share = known.groupby('Make')['Aircraft.Category'].apply(lambda s: (s == 'Airplane').mean())
known_count = known.groupby('Make').size()
airplane_makes = airplane_share[(airplane_share >= 0.9) & (known_count >= 5)].index

missing_cat = df['Aircraft.Category'].isna()
print('Missing categories before imputation:', missing_cat.sum())
df.loc[missing_cat & df['Make'].isin(airplane_makes), 'Aircraft.Category'] = 'Airplane'
print('Missing categories after imputation: ', df['Aircraft.Category'].isna().sum())

# Keep airplanes only
df = df[df['Aircraft.Category'] == 'Airplane'].copy()
print(df.shape)

Missing categories before imputation: 51513
Missing categories after imputation:  7630


(65327, 32)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

**Assumptions**
- The four injury columns together count everyone on board, so **total passengers = fatal + serious + minor + uninjured**.
- When some injury columns are filled in and others are blank, the blank ones are treated as **0** (no one in that category).
- Events where the total is still 0 have no usable occupant information and are removed.
- **`Frac.Serious.Fatal`** = (fatal + serious injuries) ÷ total passengers: the share of people on board who were seriously or fatally injured. This lets accidents on a 4-seat plane and a 300-seat plane be compared fairly.

In [12]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
print(df[injury_cols].isna().sum())

# Treat missing counts as zero, then estimate the number of people on board
df[injury_cols] = df[injury_cols].fillna(0)
df['Total.Passengers'] = df[injury_cols].sum(axis=1)
print('Events with no occupant information:', (df['Total.Passengers'] == 0).sum())
df = df[df['Total.Passengers'] > 0].copy()

# Derived metrics
df['Fatal.Serious.Injuries'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']
df['Frac.Serious.Fatal'] = df['Fatal.Serious.Injuries'] / df['Total.Passengers']
df[['Total.Passengers', 'Fatal.Serious.Injuries', 'Frac.Serious.Fatal']].describe()

Total.Fatal.Injuries      8361
Total.Serious.Injuries    9275
Total.Minor.Injuries      8793
Total.Uninjured           3946
dtype: int64
Events with no occupant information: 1111


,Total.Passengers,Fatal.Serious.Injuries,Frac.Serious.Fatal
count,64216.000000,64216.000000,64216.000000
mean,6.930858,0.838498,0.269845
std,30.514010,5.914059,0.428385
min,1.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000
75%,3.000000,1.000000,0.666667
max,699.000000,349.000000,1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [13]:
df['Aircraft.damage'].value_counts(dropna=False)

Aircraft.damage
Substantial    47369
Destroyed      13048
NaN             1873
Minor           1852
Unknown           74
Name: count, dtype: int64

**Cleaning tasks**
- `Unknown` and missing damage values carry no information about the outcome, so those rows are removed.
- **`Destroyed`** is a new 0/1 column: 1 if the aircraft was destroyed, 0 if the damage was substantial or minor. Its mean for a group is the **destruction rate**.

In [14]:
df = df[df['Aircraft.damage'].isin(['Destroyed', 'Substantial', 'Minor'])].copy()
df['Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)
print(df.shape)
df['Destroyed'].value_counts(normalize=True).round(3)

(62269, 36)


Destroyed
0    0.79
1    0.21
Name: proportion, dtype: float64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

**Make cleaning tasks** (carried out in the filtering section, because the category imputation depends on clean manufacturer names)
1. **Capitalization:** `CESSNA`, `Cessna` and `cessna` were the same manufacturer, so all names were converted to title case.
2. **Punctuation and company suffixes:** variants like `Piper Aircraft Inc`, `Piper Aircraft Corporation` and `Cessna Aircraft Co.` were reduced to the base name by removing punctuation and words such as Inc, Corp, Co, Aircraft and Aviation.
3. **Spelling variants:** remaining duplicates were mapped by hand, e.g. `Dehavilland` to `De Havilland`, `Beechcraft` to `Beech`, `Gates Learjet` to `Learjet`, `Diamond Ind` to `Diamond`.
4. **Sample size:** manufacturers with fewer than **50** recorded accidents are removed below, so every comparison rests on enough data.

In [15]:
# Accident counts per manufacturer after standardizing names
make_counts = df['Make'].value_counts()
make_counts.head(20)

Make
Cessna            25434
Piper             13990
Beech              5033
Mooney             1304
Boeing             1302
Grumman            1056
Bellanca            974
Air Tractor         890
Aeronca             605
Maule               570
Champion            501
De Havilland        460
Cirrus              449
Stinson             421
Aero Commander      405
Luscombe            386
Taylorcraft         375
North American      368
Rockwell            334
Ayres               274
Name: count, dtype: int64

In [16]:
# Keep manufacturers with at least 50 accidents
df = df[df['Make'].isin(make_counts[make_counts >= 50].index)].copy()
print('Manufacturers kept:', df['Make'].nunique())
print(df.shape)

Manufacturers kept: 58
(59447, 36)


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [17]:
# Remove rows with no model
print('Missing models:', df['Model'].isna().sum())
df = df.dropna(subset=['Model']).copy()

# Standardize model labels: upper case, no spaces or hyphens ('PA-28-140', 'PA 28 140' and 'pa28140' match)
df['Model'] = df['Model'].str.upper().str.strip().str.replace(r'[\s\-]', '', regex=True)
df['Model'].value_counts().head(10)

Missing models: 13


Model
152        2211
172        1614
172N       1088
PA28140     925
172M        786
150         742
172P        668
PA18        663
PA18150     618
182         608
Name: count, dtype: int64

In [18]:
# Are model labels unique to one manufacturer? Count models that appear under more than one Make
makes_per_model = df.groupby('Model')['Make'].nunique()
shared = makes_per_model[makes_per_model > 1]
print('Model labels used by more than one manufacturer:', len(shared))
shared.sort_values(ascending=False).head(10)

Model labels used by more than one manufacturer: 189


Model
112      5
500      5
AA5      5
AA5B     5
690B     5
AA1A     5
G164A    5
700      4
690A     4
7AC      4
Name: Make, dtype: int64

Model labels are **not** unique to each manufacturer (for example, a label like `100` or `B200` can belong to different companies). A combined **`Make.Model`** column is therefore created as a unique identifier for each plane type.

In [19]:
df['Make.Model'] = df['Make'] + ' ' + df['Model']
print('Unique plane types:', df['Make.Model'].nunique())
df['Make.Model'].value_counts().head(10)

Unique plane types: 3325


Make.Model
Cessna 152       2211
Cessna 172       1614
Cessna 172N      1088
Piper PA28140     925
Cessna 172M       786
Cessna 150        742
Cessna 172P       668
Piper PA18        663
Piper PA18150     618
Cessna 182        608
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [20]:
for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(df[col].value_counts(dropna=False), '\n')

Engine.Type
Reciprocating      51762
NaN                 3033
Turbo Prop          2620
Turbo Fan           1181
Unknown              478
Turbo Jet            337
Turbo Shaft           21
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64 

Weather.Condition
VMC    52211
IMC     4738
NaN     1808
UNK      537
Unk      140
Name: count, dtype: int64 

Number.of.Engines
1.0    48603
2.0     7925
NaN     2323
4.0      264
3.0      210
0.0      109
Name: count, dtype: int64 

Purpose.of.flight
Personal                     35000
Instructional                 8058
Unknown                       3709
Aerial Application            3307
Business                      2940
NaN                           2731
Positioning                   1013
Ferry                          550
Other Work Use                 517
Aerial Observation             379
Executive/corporate            354
Public Aircraft                311
Skydiving                      171
Flight Test                

**Cleaning tasks for the other columns**
- **Weather.Condition:** `UNK` and `Unk` are placeholders for unknown, so they become NaN. Values are upper-cased so only `VMC` (visual conditions) and `IMC` (instrument conditions) remain.
- **Engine.Type:** `Unknown`, `UNK`, `NONE` and `LR` are placeholders or errors, so they become NaN. `Geared Turbofan` (a handful of rows) is a type of turbofan and is merged into `Turbo Fan`.
- **Number.of.Engines:** an airplane cannot have **0** engines, so 0 is treated as a missing value.
- **Purpose.of.flight:** `Unknown` becomes NaN; spelling variants are merged (`Air Race/show` and `ASHO` into `Air Race show`; `PUBS` and the Federal/State/Local variants into `Public Aircraft`). Purposes with fewer than 50 events are grouped into `Other` so each category has enough data.
- **Broad.phase.of.flight:** `Unknown` and `Other` carry no information about the phase, so they become NaN.

Missing values here are left as NaN rather than filled in. They are simply excluded when each factor is analyzed.

In [21]:
# Weather
df['Weather.Condition'] = df['Weather.Condition'].str.upper().replace({'UNK': np.nan})

# Engine type
df['Engine.Type'] = df['Engine.Type'].replace({'Unknown': np.nan, 'UNK': np.nan, 'NONE': np.nan,
                                               'LR': np.nan, 'Geared Turbofan': 'Turbo Fan'})

# Number of engines: 0 is not possible for an airplane
df['Number.of.Engines'] = df['Number.of.Engines'].replace({0: np.nan})

# Purpose of flight
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({
    'Unknown': np.nan, 'Air Race/show': 'Air Race show', 'ASHO': 'Air Race show',
    'PUBS': 'Public Aircraft', 'Public Aircraft - Federal': 'Public Aircraft',
    'Public Aircraft - State': 'Public Aircraft', 'Public Aircraft - Local': 'Public Aircraft'})
purpose_counts = df['Purpose.of.flight'].value_counts()
rare_purposes = purpose_counts[purpose_counts < 50].index
df.loc[df['Purpose.of.flight'].isin(rare_purposes), 'Purpose.of.flight'] = 'Other'

# Phase of flight
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace({'Unknown': np.nan, 'Other': np.nan})

for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(df[col].value_counts(dropna=False), '\n')

Engine.Type
Reciprocating    51762
NaN               3512
Turbo Prop        2620
Turbo Fan         1182
Turbo Jet          337
Turbo Shaft         21
Name: count, dtype: int64 

Weather.Condition
VMC    52211
IMC     4738
NaN     2485
Name: count, dtype: int64 

Number.of.Engines
1.0    48603
2.0     7925
NaN     2432
4.0      264
3.0      210
Name: count, dtype: int64 

Purpose.of.flight
Personal               35000
Instructional           8058
NaN                     6440
Aerial Application      3307
Business                2940
Positioning             1013
Ferry                    550
Other Work Use           517
Public Aircraft          397
Aerial Observation       379
Executive/corporate      354
Skydiving                171
Flight Test              123
Banner Tow                97
Other                     88
Name: count, dtype: int64 

Broad.phase.of.flight
NaN            16552
Landing        11890
Takeoff         8876
Cruise          7098
Maneuvering     4625
Approach        45

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [22]:
# Keep only columns with more than 20,000 non-null values
non_null = df.notna().sum()
print('Columns dropped:', list(non_null[non_null <= 20000].index))
df = df.loc[:, non_null > 20000]
print(df.shape)
df.info()

Columns dropped: ['FAR.Description', 'Schedule', 'Air.carrier']
(59434, 34)


<class 'pandas.DataFrame'>
Index: 59434 entries, 3601 to 88886
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                59434 non-null  str           
 1   Investigation.Type      59434 non-null  str           
 2   Accident.Number         59434 non-null  str           
 3   Event.Date              59434 non-null  datetime64[us]
 4   Location                59406 non-null  str           
 5   Country                 59346 non-null  str           
 6   Latitude                22182 non-null  str           
 7   Longitude               22176 non-null  str           
 8   Airport.Code            36292 non-null  str           
 9   Airport.Name            37981 non-null  str           
 10  Injury.Severity         59434 non-null  str           
 11  Aircraft.damage         59434 non-null  str           
 12  Aircraft.Category       59434 non-null  str           
 13 

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [23]:
df.to_csv('data/AviationData_Cleaned.csv', index=False)
print('Saved', df.shape[0], 'rows and', df.shape[1], 'columns to data/AviationData_Cleaned.csv')

Saved 59434 rows and 34 columns to data/AviationData_Cleaned.csv
